# Tag 3 - Data Preparation

## Ladezelle

In [19]:
import sqlite3
import pandas as pd
from sklearn.model_selection import train_test_split

SQL_PATH = "../data/interim/projekt.sqlite"

con = sqlite3.connect(SQL_PATH)
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", con)
blockgroups = pd.read_sql("SELECT * FROM blockgruppen", con)
X = blockgroups.drop(columns=["MedHouseVal"])
y = blockgroups["MedHouseVal"]

print(blockgroups.shape)
# -> (20640, 9)

(20640, 9)


## Train-Test-Split


In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Trainingszeilen:", len(X_train))
print("Testzeilen:", len(X_test))

Trainingszeilen: 16512
Testzeilen: 4128


## Gedeckelte Werte

In [21]:
grenze_medhouseval = 5.00001
grenze_houseage = 52.0
anzahl_medhouseval = (y_train >= grenze_medhouseval).sum()
anzahl_houseage = (X_train["HouseAge"] >= grenze_houseage).sum()

print("Gedeckelte Werte aus MedHouseVal:", anzahl_medhouseval)
print("Gedeckelte Werte aus HouseAge:", anzahl_houseage)

Gedeckelte Werte aus MedHouseVal: 786
Gedeckelte Werte aus HouseAge: 1023


## Ausreißer

In [22]:
train = pd.concat([X_train, y_train], axis=1)
columns = train.select_dtypes(include="number")
q1 = 0
q3 = 0
iqr = 0

# display(columns)
# print(columns.shape)
for column in columns:
    print(f"---{column}---")
    q1 = columns[column].quantile(0.25)
    q3 = columns[column].quantile(0.75)
    iqr = q3 - q1
    untergrenze = q1 - 1.5 * iqr
    obergrenze = q3 + 1.5 * iqr
    ausreisser = columns[(columns[column] < untergrenze) |
                          (columns[column] > obergrenze)]

    print(f"Q1: {q1:.2f}")
    print(f"Q3: {q3:.2f}")
    print(f"Anzahl Ausreißer: {len(ausreisser)}")
    print(f"Anteil: {len(ausreisser) / len(columns) * 100:.2f} %\n")

---MedInc---
Q1: 2.57
Q3: 4.77
Anzahl Ausreißer: 529
Anteil: 3.20 %

---HouseAge---
Q1: 18.00
Q3: 37.00
Anzahl Ausreißer: 0
Anteil: 0.00 %

---AveRooms---
Q1: 4.45
Q3: 6.06
Anzahl Ausreißer: 410
Anteil: 2.48 %

---AveBedrms---
Q1: 1.01
Q3: 1.10
Anzahl Ausreißer: 1153
Anteil: 6.98 %

---Population---
Q1: 789.00
Q3: 1726.00
Anzahl Ausreißer: 955
Anteil: 5.78 %

---AveOccup---
Q1: 2.43
Q3: 3.28
Anzahl Ausreißer: 582
Anteil: 3.52 %

---Latitude---
Q1: 33.93
Q3: 37.72
Anzahl Ausreißer: 0
Anteil: 0.00 %

---Longitude---
Q1: -121.81
Q3: -118.01
Anzahl Ausreißer: 0
Anteil: 0.00 %

---MedHouseVal---
Q1: 1.20
Q3: 2.65
Anzahl Ausreißer: 870
Anteil: 5.27 %



# Tag 4: Baseline und erstes Modell

## Erstes Modell

In [23]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression

dummy = DummyRegressor(strategy="mean")
linear = LinearRegression()
dummy.fit(X_train, y_train)
linear.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)
y_pred_linear = linear.predict(X_test)
 
print("Erste 5 Vorhersagen Dummy:")
print(y_pred_dummy[:5])
 
print("Erste 5 Vorhersagen Linear Regression:")
print(y_pred_linear[:5])

Erste 5 Vorhersagen Dummy:
[2.07194694 2.07194694 2.07194694 2.07194694 2.07194694]
Erste 5 Vorhersagen Linear Regression:
[0.71912284 1.76401657 2.70965883 2.83892593 2.60465725]


## Metriken berechnen

In [24]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

modelle = {
    "DummyRegressor": dummy,
    "LinearRegression": linear
}
ergebnisse = []

for name, modell in modelle.items():
    y_pred = modell.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    ergebnisse.append({
        "Modell": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })
# -----------------------------------------------------------------------
# | Metrik | Was misst sie?	                                  | Optimal |
# | MAE	   | Durchschnittliche absolute Abweichung	          | 0       |
# | RMSE   | Fehlermaß mit stärkerer Gewichtung großer Fehler | 0       |
# | R²	   | Modellgüte relativ zur Mittelwertvorhersage	  | 1       |
# -----------------------------------------------------------------------

vergleich = pd.DataFrame(ergebnisse)
display(vergleich.round(4))

,Modell,MAE,RMSE,R2
0,DummyRegressor,0.9061,1.1449,-0.0002
1,LinearRegression,0.5332,0.7456,0.5758


## Prozentuale Verbesserung

In [28]:
mae_dummy = vergleich.loc[vergleich["Modell"] == "DummyRegressor", "MAE"].iloc[0]
mae_linear = vergleich.loc[vergleich["Modell"] == "LinearRegression", "MAE"].iloc[0]

verbesserung = (mae_dummy - mae_linear) / mae_dummy * 100
mae_usd = mae_linear * 100_000

print(f"Verbesserung gegenüber der Baseline: {verbesserung:.2f}%")
print(f"MAE der LinearRegression: {mae_usd:,.2f} USD")

Verbesserung gegenüber der Baseline: 41.15%
MAE der LinearRegression: 53,320.01 USD
